# Rubric-Based Reinforcement Learning for Structured Reasoning in Gemma
This notebook fine-tunes Gemma-2-2B-IT to reliably produce structured outputs in the form:
`
<reasoning>...</reasoning><answer>...</answer>
`
The goal is to improve format compliance, answer correctness, and overall response quality using a two-stage pipeline that is reproducible on a single Kaggle TPU v5e-8 session (with checkpointing for multi-session runs).

## Overall training and evaluation strategy
### Stage 1 — Supervised Fine-Tuning (SFT)

We first teach the model the desired output structure using instruction-style data that includes reasoning traces and final answers. This establishes a strong baseline that follows the required tag format.


### Stage 2 — GRPO (Reinforcement Learning)

After SFT establishes the basic `<reasoning>...<answer>` structure, we apply **Group Relative Policy Optimization (GRPO)** to refine both reasoning quality and answer correctness. GRPO optimizes the model by comparing outputs within groups rather than individual samples, enabling stable training with our multi-faceted reward signal.

---

### Compute allocation strategy
We trained entirely on Kaggle TPU v5e-8. To maximize available compute:
- We used QLoRA to reduce memory footprint during SFT.
- SFT consumed a small fraction of compute (1 epoch) to learn output format.
- GRPO consumed the majority of TPU time for reward optimization.
- We split the TPU mesh during GRPO:
  - (1,4) devices for policy + reference
  - (1,4) devices for judge model
- This enabled parallel rollouts and judge scoring, avoiding sequential bottlenecks.
- This layout allowed us to run RL with a Gemma-2-2B policy and a Gemma-3-12B judge entirely within Kaggle resource constraints.

---

### Reward setup for RL
#### 1. Format Reward
- Binary reward indicating correct `<reasoning>...</reasoning><answer>...</answer>` structure.

#### 2. Exact Answer Reward
- Binary numeric match reward for tasks with verifiable answers.

#### 3. G-RaR (Rubric-Guided LLM-as-Judge Reward)
A continuous reward computed via a **Gemma-3-12B judge** using:
- task type
- task-specific rubrics
- softmax expectation over discrete `{1..5}` rating tokens

This produces a **0–1 normalized continuous signal**.
We used **group-relative advantage** to stabilize GRPO and reduce variance across heterogeneous tasks.


---
### Evaluation strategy
Evaluation strategy
We tracked four evaluation axes:
- Format accuracy
- Exact match accuracy
- Partial numeric match
- Rubric-based reasoning score (via G-RaR)

## Dataset
We train on our custom TFDS dataset reasoning_from_hf, designed as a dual-purpose corpus: an SFT split to teach structured reasoning traces, and an RL split to support GRPO with rubric-based rewards.
### 1.Overview
| Split | Count | Role | Key Features |
| --- | --- | --- | --- |
| **SFT** | 33,142 | Reasoning Baseline | OpenThoughts (12k) + Chimbiwide Creative (10k) + Chimbiwide General (11k) |
| **RL** | 36,250 | Reward Signal | GSM8K (4k) + Code Contests (4k) + SciQ (4k) + WritingPrompts (10k) + XSum (10k) + Dolly (4.7k) |
| **Total** | **69,392** | - | - |

Public access: The final dataset is published on Hugging Face Hub(https://huggingface.co/datasets/comoZ/reasoning-dataset) and mirrored as a Kaggle TFDS dataset(https://www.kaggle.com/datasets/hojuna/comoz-tfds) for evaluation.

---

### 2. Record schema
Below is the schema we expose in TFDS / preprocessing
| Field       | Description                                 |
| ----------- | ------------------------------------------- |
| `input`     | original question / prompt                  |
| `output`    | ground-truth answer (when available)        |
| `think`     | gold reasoning trace        |
| `source`    | data provenance                             |
| `type`      | high-level category label                   |
| `task_type` | task schema used for evaluation routing     |
| `rubrics`   | task-specific rubric text used by the judge |

---

### 3. Data genration steps
#### Collection (task coverage)
We intentionally mix verifiable and non-verifiable tasks so GRPO can learn both answer correctness and reasoning quality.

- Verifiable: GSM8K (math), Code Contests (coding), SciQ (science)

- Non-verifiable: WritingPrompts (writing), XSum (summarization), Dolly (instruction following)

#### Splitting (SFT vs RL)

- SFT split: includes explicit thought processes to teach the <reasoning>...</reasoning> style output.

- RL split: built to support GRPO reward computation across diverse tasks with pre-generated rubrics.

#### Rubric generation (offline, for GRPO efficiency)
To avoid generating rubrics online during RL, we pre-compute metadata:

- We run an offline “metadata augmentation pipeline” to attach evaluation criteria.
- Using Gemma-3-27B-IT, we pre-generate 3-tier customized rubrics per question and store them in the dataset.
####
This makes GRPO faster because the judge can score immediately without extra rubric-generation compute

# Pre-stage: Environment Settings

## Install Necessary Packages

In [ ]:
!pip install --upgrade pip

!pip install -q kagglehub
!pip install -q ipywidgets

!pip install -q tensorflow
!pip install -q tensorflow_datasets
!pip install -q tensorboardX
!pip install -q transformers
!pip install -q grain
!pip install "google-tunix[prod]==0.1.3"

!pip uninstall -q -y flax
!pip install -q  wandb==0.22.0
!pip install --upgrade qwix

## Import Packages

In [ ]:
# Set environment variables
import os
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

In [ ]:
# System packages
import re
import gc
import time
import functools
from pprint import pprint
from pathlib import Path
from tqdm.auto import tqdm

# Tools
import wandb
import humanize
import numpy as np
from sklearn.model_selection import train_test_split

# Google JAX
import jax
import jax.numpy as jnp
from jax.sharding import Mesh
from flax import nnx
import qwix
import optax
import orbax.checkpoint as ocp
import grain.python as grain

# Google Tunix
from tunix import PeftTrainer, TrainingConfig, MetricsLoggerOptions
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib
from tunix.models.gemma import params_safetensors as params_safetensors_lib
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner
from tunix.rl.rollout import base_rollout
from tunix.sft import metrics_logger
from tunix.sft.peft_trainer import TrainingInput
from tunix.sft import utils

# Kaggle
import kagglehub
from kaggle_secrets import UserSecretsClient
import tensorflow_datasets as tfds

In [ ]:
# Check JAX setting
print(f"JAX version: {jax.__version__}")
print(f"Number of devices: {len(jax.devices())}")
print(f"Device kind: {jax.devices()[0].device_kind}")
print(f"JAX backend: {jax.default_backend()}")
print(f"\nDevices:")
for i, device in enumerate(jax.devices()):
    print(f"  [{i}] {device}")
print("="*60)

if jax.default_backend() != 'tpu':
    print("\n⚠️  WARNING: Not running on TPU!")
    print(f"   Current backend: {jax.default_backend()}")
    print("   Make sure you've selected TPU runtime in Kaggle")
else:
    print("\n✓ TPU backend confirmed")

os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ['KAGGLE_USERNAME'] = UserSecretsClient().get_secret("KAGGLE_USERNAME")
os.environ['KAGGLE_KEY'] = UserSecretsClient().get_secret("KAGGLE_KEY")

jax.config.update('jax_enable_x64', False)
jax.config.update('jax_default_matmul_precision', 'high')

## Utility Functions

In [ ]:
# Global Hyperparameters
## LoRA
RANK = 16
ALPHA = 32
QUANTIZATION = True

## Dataset
DATASET_DIR = "/kaggle/input/comoz-tfds"
DATASET_NAME = "reasoning_from_hf"

reasoning_start = "<reasoning>"
reasoning_end = "</reasoning>"
solution_start = "<answer>"
solution_end = "</answer>"

SYSTEM_PROMPT = f"""You are given a problem. Think about the problem and provide your reasoning.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide the final answer (i.e., just one numerical value) between {solution_start} and {solution_end}."""

TEMPLATE = """<start_of_turn>user
{system_prompt}

{question}<end_of_turn>
<start_of_turn>model"""

In [ ]:
def get_lora_model(base_model, mesh, quantize=False):
    if quantize:
        lora_provider = qwix.LoraProvider(
            module_path=(
                ".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj|"
                ".*attn_vec_einsum"
            ),
            rank=RANK,
            alpha=ALPHA,
            weight_qtype="nf4",
        )
    else:
        lora_provider = qwix.LoraProvider(
            module_path=(
                ".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj|"
                ".*attn_vec_einsum"
            ),
            rank=RANK,
            alpha=ALPHA,
        )

    model_input = base_model.get_model_input()
    lora_model = qwix.apply_lora_to_model(
        base_model, lora_provider, **model_input
    )

    print("\nSharding model across TPU devices...")
    with mesh:
        state = nnx.state(lora_model)
        pspecs = nnx.get_partition_spec(state)

        # Force materialization on TPU
        sharded_state = jax.tree_util.tree_map(
            lambda x, p: jax.lax.with_sharding_constraint(x, p),
            state, pspecs
        )
        nnx.update(lora_model, sharded_state)

    return lora_model

In [ ]:
def _as_text(v):
    if v is None:
        return ""
    if hasattr(v, "numpy"):
        v = v.numpy()
    if isinstance(v, bytes):
        return v.decode("utf-8")
    return str(v)

def tfds_to_grain(tfds_ds):
    def gen():
        for x in tfds_ds:
            yield {k: _as_text(v) for k, v in x.items()}
    return grain.IterDataset.source(gen)

class TFDSWrapper:
    """
    Wrapper to satisfy grain.MapDataset source constraints in the Kaggle environment.
    """
    def __init__(self, tfds_ds):
        self.data = [
            {k: _as_text(v) for k, v in x.items()}
            for x in tfds_ds
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [ ]:
def transform_fn(x):
    """
    Convert TFDS column schema into the model-required key structure -> generates prompt
    """
    input_text = _as_text(x.get("input", ""))
    output_text = _as_text(x.get("output", ""))
    think_text = _as_text(x.get("think", ""))
    source_text = _as_text(x.get("source", ""))
    category = _as_text(x.get("type", ""))
    task_type = _as_text(x.get("task_type", ""))
    rubrics = _as_text(x.get("rubrics", ""))

    return {
        "prompts": TEMPLATE.format(
            system_prompt=SYSTEM_PROMPT,
            question=input_text,
        ),
        "question": input_text,
        "output": output_text,
        "think": think_text,
        "source": source_text,
        "category" : category,
        "task_type" : task_type,
        "rubrics" : rubrics

    }


def get_dataset(split="rl") -> grain.MapDataset:
    if split not in ["rl", "sft"]:
        raise ValueError(f"split must be 'rl' or 'sft', got: {split}")

    print(f"[INFO] loading TFDS dataset: {DATASET_NAME}, split={split}")

    tfds_ds = tfds.load(
        DATASET_NAME,
        data_dir=DATASET_DIR,
        split=split,
    )

    if split == "rl":
        tfds_ds = tfds_ds.shuffle(buffer_size=10_000, seed=42)

    wrapped = TFDSWrapper(tfds_ds)
    dataset = grain.MapDataset.source(wrapped)
    dataset = dataset.map(transform_fn)

    return dataset

In [ ]:
# show_hbm_usage helps to monitor VRAM
def show_hbm_usage():
  """Displays memory usage per device."""
  fmt_size = functools.partial(humanize.naturalsize, binary=True)

  for d in jax.local_devices():
    stats = d.memory_stats() # Get current memory statistic information of selected device(d) in dictionary form
    used = stats["bytes_in_use"]
    limit = stats["bytes_limit"]
    print(f"Using {fmt_size(used)} / {fmt_size(limit)} ({used/limit:%}) on {d}")

# Stage 1 — Supervised Fine-Tuning (SFT)

Directly applying GRPO to a base model presents several critical challenges. First, the model lacks exposure to the `<reasoning>...<answer>` format, making reward computation unreliable—format verification rewards would be consistently zero, and answer extraction would fail entirely. Second, without structured outputs, the policy gradient estimates become extremely noisy, as the model explores a vast unstructured action space before discovering the desired format. Third, cold-starting GRPO requires the model to simultaneously learn both the output structure and the reasoning quality, leading to slow convergence and potential collapse to degenerate solutions.

SFT addresses these issues by establishing a strong initialization for GRPO. By training on high-quality reasoning traces, the model learns to:

 1. Consistently produce the required tag structure, enabling reliable reward computation
 2. Generate coherent reasoning steps before final answers, providing a meaningful foundation for quality refinement
 3. Start GRPO training from a distribution close to the desired output space, dramatically improving sample efficiency

In essence, SFT creates a "warm start" for reinforcement learning—the model already knows how to format outputs correctly and produce reasonable reasoning, allowing GRPO to focus on refining quality rather than discovering basic structure.

## 1-1. Hyperparameters


In [ ]:
MESH_SHAPE = (2, 4)
KAGGLE_MODEL_NAME = "google/gemma-2/transformers/gemma-2-2b-it"

# Optimizer
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPSILON = 1e-8
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Train
MAX_SEQ_LENGTH = 4096

TRAIN_MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 2e-5
NUM_EPOCHS = 1

# Logging
CHECKPOINT_DIR = "/kaggle/working/outputs_sft_full/checkpoints"
TENSORBOARD_DIR = "/kaggle/working/outputs_sft_full/tensorboard"
SAVE_INTERVAL_STEPS = 3000
EVAL_INTERVAL_STEPS = 3000
LOG_INTERVAL_STEPS = 1


print(f"Global Batch Size: {TRAIN_MICRO_BATCH_SIZE * 8 * GRADIENT_ACCUMULATION_STEPS}")
print(f"Total Training Epochs: {NUM_EPOCHS}")

## 1-2. Load Model
For SFT, we use quantize `gemma-2-2b-it` model with LoRA adapters.


In [ ]:
print(f"Model Name or Path: {KAGGLE_MODEL_NAME}")

local_model_path = kagglehub.model_download(KAGGLE_MODEL_NAME)
print(f"✓ Model downloaded to: {local_model_path}")

print(f"\nCreating TPU mesh with shape {MESH_SHAPE}...")
mesh = jax.make_mesh(
    MESH_SHAPE, ('fsdp', 'tp')
)

print(f"✓ TPU Mesh created successfully")
print(f"  Mesh shape: {mesh.shape}")
print(f"  Mesh axis names: {mesh.axis_names}")

In [ ]:
model_config = gemma_lib.ModelConfig.gemma2_2b()

with mesh:
    base_model = params_safetensors_lib.create_model_from_safe_tensors(
        local_model_path,  # Directory containing .safetensors files
        model_config,
        mesh,
    )
    nnx.display(base_model)
print("✓ Model loaded successfully")


tokenizer = tokenizer_lib.Tokenizer(
    tokenizer_path=f"{local_model_path}/tokenizer.model"
)
print("✓ Tokenizer loaded successfully")

In [ ]:
gemma2_model = get_lora_model(
    base_model=base_model,
    mesh=mesh,
    quantize=QUANTIZATION,
)
print("✓ LoRA Adapter loaded successfully")
nnx.display(gemma2_model)

## 1-3. Create Dataset

In [ ]:
def example_formatting(example):
    question = example['question']
    answer = example['output']
    reasoning = example['think']

    # User Turn
    text = f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{question}<end_of_turn>\n"

    # Model Turn
    text += f"<start_of_turn>model\n"
    text += f"<reasoning>\n{reasoning}\n</reasoning>\n"
    text += f"<answer>\n{answer}\n</answer>"
    text += f"<end_of_turn>"

    return {"text": text}


def tokenize_function(example):
    full_text = example["text"]
    full_tokens = tokenizer.encode(full_text)

    prompt_text = full_text.split("<start_of_turn>model")[0] + "<start_of_turn>model\n"
    prompt_tokens = tokenizer.encode(prompt_text)
    prompt_len = len(prompt_tokens)

    # Padding/Truncation Logic
    if len(full_tokens) > MAX_SEQ_LENGTH:
        full_tokens = full_tokens[:MAX_SEQ_LENGTH]
    else:
        pad_token = tokenizer.pad_id() if hasattr(tokenizer, 'pad_id') else tokenizer.eos_id()
        full_tokens = full_tokens + [pad_token] * (MAX_SEQ_LENGTH - len(full_tokens))

    input_tokens = np.array(full_tokens, dtype=np.int32)

    # Create Mask
    loss_mask = np.zeros_like(input_tokens, dtype=np.float32)

    # Enable loss only for the response part (ignoring padding)
    seq_len = min(len(tokenizer.encode(full_text)), MAX_SEQ_LENGTH)
    if seq_len > prompt_len:
        loss_mask[prompt_len:seq_len] = 1.0

    return TrainingInput(input_tokens=input_tokens, input_mask=loss_mask)

In [ ]:
try:
    sft_dataset = get_dataset("sft")
except Exception as e:
    print(f"✓ Dataset preparation failed: {e}")

data_list = list(sft_dataset)

train_dataset, test_dataset = train_test_split(data_list, test_size=0.1, random_state=42)

formatted_train = [example_formatting(ex) for ex in train_dataset]
formatted_test = [example_formatting(ex) for ex in test_dataset]

# Create Grain datasets
train_grain = (
    grain.MapDataset.source(formatted_train)
    .map(tokenize_function)
    .shuffle(seed=42)
    .repeat(NUM_EPOCHS)
    .batch(batch_size=TRAIN_MICRO_BATCH_SIZE, drop_remainder=True)
)

eval_grain = (
    grain.MapDataset.source(formatted_test)
    .map(tokenize_function)
    .batch(batch_size=TRAIN_MICRO_BATCH_SIZE, drop_remainder=True)
)

print(f"✓ Train batches: {len(train_grain):,}")
print(f"✓ Eval batches: {len(eval_grain):,}")

## 1-4. Train Model

### Generate Training Config

In [ ]:
MAX_STEPS = len(train_grain) // GRADIENT_ACCUMULATION_STEPS
WARMUP_STEPS = int(MAX_STEPS * 0.05)


checkpointing_options = ocp.CheckpointManagerOptions(
    save_interval_steps=SAVE_INTERVAL_STEPS,
    max_to_keep=3,
)

training_config = TrainingConfig(
    max_steps=MAX_STEPS,
    eval_every_n_steps=EVAL_INTERVAL_STEPS,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    checkpoint_root_directory=CHECKPOINT_DIR,
    checkpointing_options=checkpointing_options,
    metrics_logging_options=MetricsLoggerOptions(
        log_dir=TENSORBOARD_DIR,
        flush_every_n_steps=LOG_INTERVAL_STEPS
    ),
)

print("✓ Training configuration created")
print(f"  Max steps: {MAX_STEPS}")
print(f"  Micro batch size: {TRAIN_MICRO_BATCH_SIZE}")
print(f"  Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch size: {TRAIN_MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Eval interval: {EVAL_INTERVAL_STEPS}")
print(f"  Save interval: {SAVE_INTERVAL_STEPS}")

### Make Scheduler & Optimizer Chain

In [ ]:
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    decay_steps=MAX_STEPS - WARMUP_STEPS,
    end_value=LEARNING_RATE * 0.1,
)

# Create optimizer chain
optimizer = optax.chain(
    optax.clip_by_global_norm(MAX_GRAD_NORM),
    optax.scale_by_adam(
        b1=ADAM_BETA1,
        b2=ADAM_BETA2,
        eps=ADAM_EPSILON,
    ),
    optax.add_decayed_weights(WEIGHT_DECAY),
    optax.scale_by_schedule(schedule),
    optax.scale(-1.0),  # Gradient descent
)

print("✓ Optimizer configured:")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print(f"  Total epochs: {NUM_EPOCHS}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Max grad norm: {MAX_GRAD_NORM}")

### Get Trainer

In [ ]:
def gen_model_input_fn(training_input):
    """Convert TrainingInput to model-compatible format."""
    pad_mask = training_input.input_tokens != 0
    positions = utils.build_positions_from_mask(pad_mask)
    attention_mask = utils.make_causal_attn_mask(pad_mask)

    return {
        'input_tokens': training_input.input_tokens,
        'input_mask': training_input.input_mask,
        'positions': positions,
        'attention_mask': attention_mask,
    }

trainer = PeftTrainer(
    model=gemma2_model,
    optimizer=optimizer,
    training_config=training_config,
)
trainer = trainer.with_gen_model_input_fn(gen_model_input_fn)

print("✓ Trainer ready for training")
print(f"  Model: Gemma 2 2B (Quantize LoRA)")
print(f"  Max steps: {MAX_STEPS}")

### Training Model

In [ ]:
print("="*60)
print("Starting Full Fine-Tuning on TPU v5e-8")
print("="*60)
print(f"Max steps: {MAX_STEPS}")
print(f"Training examples: {len(formatted_train)}")
print(f"Eval examples: {len(formatted_test)}")
print(f"Batch size: {TRAIN_MICRO_BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print("="*60)

all_params = nnx.state(gemma2_model)
param_leaves = jax.tree_util.tree_leaves(all_params)
if len(param_leaves) > 0:
    sample_param = param_leaves[0]
    if hasattr(sample_param, 'devices'):
        devices = sample_param.devices()
        if len(devices) > 0:
            device_kind = list(devices)[0].device_kind
            print(f"✓ Model parameters are on: {device_kind}")
            if 'tpu' not in device_kind.lower():
                print(f"⚠️  WARNING: Model params on {device_kind}, not TPU!")
                print(f"⚠️  Training will run on CPU and produce wrong results!")
            else:
                print(f"✓✓✓ CONFIRMED: Model is ready for TPU training!")
        else:
            print("⚠️  No devices found for model parameters")
    else:
        print("⚠️  Cannot check device placement")
else:
    print("⚠️  No model parameters found")
print("="*60)

In [ ]:
print("\n" + "="*60)
print("IMPORTANT: First training step will take 5-10 minutes")
print("="*60)
print("JAX is compiling all functions (happens on CPU).")
print("After first step completes, TPU will be used and steps will be MUCH faster.")
print("You should see 'Compiling...' messages initially.")
print("="*60)

print("\nStarting training...")
start_time = time.time()

trainer.train(
    train_ds=train_grain,
    eval_ds=eval_grain,
)

end_time = time.time()
total_time = end_time - start_time

print("\n" + "="*60)
print("Training Completed!")
print("="*60)
print(f"Total training time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"Average time per step: {total_time/MAX_STEPS:.1f} seconds")
print(f"Checkpoints saved to: {CHECKPOINT_DIR}")
print("="*60)

print("\n" + "="*60)
print("POST-TRAINING: Verify TPU was used")
print("="*60)
print(f"Expected TPU time: 5-15 seconds per step after compilation")
print(f"Your average: {total_time/MAX_STEPS:.1f} seconds per step")
if total_time/MAX_STEPS < 1.0:
    print("❌ WARNING: Training ran on CPU, not TPU!")
    print("Results will be incorrect. Check that model is properly sharded.")
else:
    print("✓ Training timing looks correct for TPU usage!")
print("="*60)

## 1-5. Save Trained Model

In [ ]:
wandb.init(project="tunix")

SFT_CKPT_DIR = "/kaggle/working/sft_checkpoint/"
os.makedirs(SFT_CKPT_DIR, exist_ok=True)

sft_lora_state = nnx.state(gemma2_model, nnx.LoRAParam)
print(f"\nNumber of LoRA parameters to save: {len(jax.tree.leaves(sft_lora_state))}")

sft_checkpointer = ocp.StandardCheckpointer()

# Save LoRA Adapter
sft_save_path = os.path.join(SFT_CKPT_DIR, "lora_params")
print(f"\nSaving model to: {sft_save_path}")
sft_checkpointer.save(sft_save_path, sft_lora_state,force=True)

sft_checkpointer.wait_until_finished()

print("✓ SFT model saved successfully!")
print(f"   Save path: {sft_save_path}")

In [ ]:
if os.path.exists(sft_save_path):
    files = list(Path(sft_save_path).rglob("*"))
    print(f"   Number of saved files: {len(files)}")
    print("=" * 60)
else:
    print(f"   ❌ Save path not found!")
    print("=" * 60)

print("\n✓ Saved model information:")
print(f"   - Save path: {sft_save_path}")
print(f"   - Number of LoRA parameters: {len(jax.tree.leaves(sft_lora_state))}")
print("=" * 60)

In [ ]:
# Cleanup SFT resources to prevent Mesh conflict and OOM
del gemma2_model
del all_params
del param_leaves
del sft_lora_state


try:
    if 'base_model' in globals(): del base_model
    if 'mesh' in globals(): del mesh
    if 'lora_model' in globals(): del lora_model
    if 'optimizer' in globals(): del optimizer
    if 'trainer' in globals(): del trainer
    if 'tokenizer' in globals(): del tokenizer
except NameError:
    pass


gc.collect()
jax.clear_caches()
print("🧹 SFT Resources cleared.")

In [ ]:
show_hbm_usage()

---

# Stage 2 - GRPO: Group Relative Policy Optimization

After SFT establishes the basic `<reasoning>...<answer>` structure, we apply **Group Relative Policy Optimization (GRPO)** to refine both reasoning quality and answer correctness. GRPO optimizes the model by comparing outputs within groups rather than individual samples, enabling stable training with our multi-faceted reward signal.

**Why GRPO?**

We chose GRPO for three key reasons. First, it handles our composite reward function effectively by normalizing rewards within each group, making training robust to variance across different task types. Second, it reduces optimization instability compared to standard policy gradient methods by computing relative advantages rather than absolute rewards. Third, it enables efficient training within Kaggle's 9-hour TPU constraint—the group-relative approach requires fewer samples for reliable gradient estimates.

## 2-1. Hyperparameters

In [ ]:
# GRPO
NUM_ITERATIONS = 1
BETA = 0.08
EPSILON = 0.1

# Train
TRAIN_MICRO_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4

NUM_BATCHES = 1000

NUM_VADIATION_DATA = 500
EVAL_EVERY_N_STEPS = 10
NUM_EPOCHS = 1
TRAIN_FRACTION = 1.0

MAX_STEPS = int((NUM_BATCHES * NUM_ITERATIONS * TRAIN_FRACTION * NUM_EPOCHS))

INTERMEDIATE_CKPT_DIR = "/kaggle/working/intermediate_ckpt/"
RL_CKPT_DIR = "/kaggle/working/RL_checkpoints"
RL_FULL_CKPT_DIR = "/kaggle/working/RL_FULL_checkpoint"

SAVE_INTERVAL_STEPS = 500
MAX_TO_KEEP = 4

# Optimizer, Scheduler
LEARNING_RATE = 1e-5
B1 = 0.9
B2 = 0.99
WEIGHT_DECAY = 0.1

WARMUP_STEPS = 0.1 * MAX_STEPS
MAX_GRAD_NORM = 0.1

# Generation
TEMPERATURE = 0.9
TOP_P = 1.0
TOP_K = 50

"""
Conventionally, it has been assumed that a larger NUM_Generation (Group Size) yields better performance in GRPO.
However, recent research demonstrates that training can proceed efficiently even with a NUM_Generation of 2 (pairwise).
[https://arxiv.org/pdf/2510.00977]
Based on these findings, I propose setting NUM_Generation = 2 for our training to maximize computational efficiency.
"""

NUM_GENERATIONS = 2

GENERATION_CONFIGS = {
    # greedy search
    "greedy": {"temperature": 1e-4, "top_k": 1, "top_p": 1.0},
    # some randomness
    "standard": {"temperature": 0.7, "top_k": 50, "top_p": 0.95},
    # liberal
    "liberal": {"temperature": 0.85, "top_k": 2000, "top_p": 1.0},
}

TRAINER_MAX_PROMPT_LENGTH = 1024
TRAINER_TOTAL_GENERATION_STEPS =1024

## 2-2. Load Model

In the JAX and Tunix ecosystem, there are two primary ways to load pre-trained models. To demonstrate both strategies, we load the ref_model (LoRA policy) using the NNX library, while the judge_model is loaded directly via Tunix.

In [ ]:
all_devices = jax.devices()

# Allocate the first 4 devices for the Training (Actor) model.
train_device_slice = all_devices[:4]

# Allocate the remaining 4 devices (indices 4 to 7) for the Judge (Reward) model.
judge_device_slice = all_devices[4:]

# Setup Mesh for the Training / judge Model.
train_mesh_shape = (1, 4)
train_mesh = Mesh(
    np.array(train_device_slice).reshape(train_mesh_shape),
    axis_names=("fsdp", "tp")
)

judge_mesh_shape = (1, 4)
judge_mesh = Mesh(
    np.array(judge_device_slice).reshape(judge_mesh_shape),
    axis_names=("fsdp", "tp")
)

In [ ]:
ref_model_path = "google/gemma-2/flax/gemma2-2b-it"
ref_model_version = "gemma2-2b-it"
ref_param_version = "2-2b-it"

ref_kaggle_ckpt_path = kagglehub.model_download(ref_model_path)

In [ ]:
!rm /kaggle/working/intermediate_ckpt/* -rf

ref_params = params_lib.load_and_format_params(
    os.path.join(ref_kaggle_ckpt_path, ref_model_version)
)

ref_gemma = gemma_lib.Transformer.from_params(ref_params, version=ref_param_version)
ref_checkpointer = ocp.StandardCheckpointer()

_, ref_state = nnx.split(ref_gemma)
ref_checkpointer.save(os.path.join(INTERMEDIATE_CKPT_DIR, "state_ref"), ref_state, force=True)
ref_checkpointer.wait_until_finished()

del ref_params
del ref_gemma
del ref_state
gc.collect()

In [ ]:
def get_gemma_model(ckpt_path, mesh):
    model_config = gemma_lib.ModelConfig.gemma2_2b()

    abs_gemma: nnx.Module = nnx.eval_shape(
        lambda: gemma_lib.Transformer(model_config, rngs=nnx.Rngs(params=0))
    )

    abs_state = nnx.state(abs_gemma)
    abs_state = jax.tree.map(
        lambda a, s: jax.ShapeDtypeStruct(a.shape, jnp.bfloat16, sharding=s),
        abs_state,
        nnx.get_named_sharding(abs_state, mesh),
    )

    checkpointer = ocp.StandardCheckpointer()
    restored_params = checkpointer.restore(ckpt_path, target=abs_state)

    graph_def, _ = nnx.split(abs_gemma)
    gemma = nnx.merge(graph_def, restored_params)
    return gemma, mesh, model_config

In [ ]:
from tunix.models.gemma3 import params as judge_params
from tunix.models.gemma3 import model as judge_model

## load from tunix directly
## Models supported by Tunix can be loaded directly from the library.
## You can check the list of available models for the current version in the GitHub repository.
## https://github.com/google/tunix/tree/v0.1.3/tunix/models

with judge_mesh:
    JUDGE_MODEL_CP_PATH = judge_params.GEMMA3_12B_IT
    judge_model_config = judge_model.ModelConfig.gemma3_12b()
    judge_model = judge_params.create_model_from_checkpoint(JUDGE_MODEL_CP_PATH, judge_model_config, mesh=judge_mesh, dtype=jnp.bfloat16)
    judge_tokenizer = judge_params.create_tokenizer()

In [ ]:
show_hbm_usage()

In [ ]:
# Reference model
ref_model, ref_mesh, ref_model_config = get_gemma_model(
    ckpt_path=os.path.join(INTERMEDIATE_CKPT_DIR, "state_ref"), mesh = train_mesh)

## Load tokenizer
if "gemma2" in ref_model_version:
    ref_tokenizer = tokenizer_lib.Tokenizer(
        tokenizer_path=os.path.join(ref_kaggle_ckpt_path, "tokenizer.model")
    )

## 2-3. Create Dataset

Load dataset for GRPO

In [ ]:
try:
    rl_dataset = get_dataset("rl")
    for ex in rl_dataset:
        print("\n===== SAMPLE =====")
        for k, v in ex.items():
            print(f"{k}: {v}")
        break

except Exception as e:
    print(f"❌ Dataset preparation failed: {e}")

Remove data exceeding the limit. Fixed cache size is allocated during rollout, so larger inputs will cause errors.

In [ ]:
import grain.python as grain

def filter_dataset_with_tokenizer(dataset, ref_tokenizer, judge_tokenizer, max_prompt_length):

    #Calculate token lengths and extract indices (Eager Execution)
    indices_to_keep = []
    print(f"Calculating token lengths... (Threshold: {max_prompt_length})")

    for i in range(len(dataset)):

        # Retrieve data item
        item = dataset[i]
        prompt = item["prompts"]

        if i == 0:
            print(prompt)

        # Encode with tokenizer and check length
        ref_token_len = len(ref_tokenizer.encode(prompt))
        judge_token_len = len(judge_tokenizer.encode(prompt))

        if ref_token_len < max_prompt_length and judge_token_len < max_prompt_length:
            indices_to_keep.append(i)

        if (i + 1) % 5000 == 0:
            print(f"Progress: {i + 1}/{len(dataset)} finished")

    print(f"Filtering results: {len(dataset)} -> {len(indices_to_keep)} remaining")

    index_ds = grain.MapDataset.source(indices_to_keep)
    filtered_ds = index_ds.map(lambda idx: dataset[idx])

    return filtered_ds

In [ ]:
filtered_rl_dataset = filter_dataset_with_tokenizer(rl_dataset, ref_tokenizer, judge_tokenizer, TRAINER_MAX_PROMPT_LENGTH)

In [ ]:
# Check whether long data is deleted

def calculate_dataset_stats(dataset, tokenizer):
    """
    Calculates token length statistics of the dataset using the tunix tokenizer.
    """
    token_lengths = []
    char_lengths = []

    print(f"Starting dataset statistics calculation (Total samples: {len(dataset)})...")

    for i in range(len(dataset)):
        # Retrieve data (assuming transform_fn is applied)
        prompt_text = dataset[i]["prompts"]

        # 1. Store character length
        char_lengths.append(len(prompt_text))

        # 2. Tokenization and length measurement using Tunix tokenizer
        # Typically, tunix tokenizer returns a list of ids using .encode()
        tokens = tokenizer.encode(prompt_text)
        token_lengths.append(len(tokens))

        # Display progress (every 5000 steps)
        if (i + 1) % 5000 == 0:
            print(f"Progress: {i + 1}/{len(dataset)} finished")

    # Result analysis (using NumPy)
    token_np = np.array(token_lengths)
    char_np = np.array(char_lengths)

    stats = {
        "Token": {
            "Max": np.max(token_np),
            "Mean": np.mean(token_np),
            "95th": np.percentile(token_np, 95),
            "99th": np.percentile(token_np, 99),
        },
        "Char": {
            "Max": np.max(char_np),
            "Mean": np.mean(char_np),
        }
    }

    return stats

In [ ]:
# RL dataset distribution
stats = calculate_dataset_stats(rl_dataset, ref_tokenizer)

print("\n" + "="*30)
print("📊 Dataset Length Statistics (based on gemma2 tokenizer)")
print("="*30)
print(f"[Tokens] Max: {stats['Token']['Max']:.0f}")
print(f"[Tokens] 99th Percentile: {stats['Token']['99th']:.0f}")
print(f"[Tokens] 95th Percentile: {stats['Token']['95th']:.0f}")
print(f"[Tokens] Average: {stats['Token']['Mean']:.1f}")
print("-" * 30)
print(f"[Chars]  Max: {stats['Char']['Max']}")
print(f"[Chars]  Average: {stats['Char']['Mean']:.1f}")

In [ ]:
rl_dataset = filtered_rl_dataset

In [ ]:
# RL dataset distribution after filtering
stats = calculate_dataset_stats(rl_dataset, ref_tokenizer)

print("\n" + "="*30)
print(f"📊 Dataset Length Statistics (based on gemma2 tokenizer)")
print("="*30)
print(f"[Tokens] Max: {stats['Token']['Max']:.0f}")
print(f"[Tokens] 99th Percentile: {stats['Token']['99th']:.0f}")
print(f"[Tokens] 95th Percentile: {stats['Token']['95th']:.0f}")
print(f"[Tokens] Average: {stats['Token']['Mean']:.1f}")
print("-" * 30)
print(f"[Chars]  Max: {stats['Char']['Max']}")
print(f"[Chars]  Average: {stats['Char']['Mean']:.1f}")

Split and batch training and validation data. Validation data is split by a configured constant

In [ ]:
# split into train dataset and validation dataset
if NUM_VADIATION_DATA <= 0:
  train_dataset = rl_dataset.batch(TRAIN_MICRO_BATCH_SIZE)[:NUM_BATCHES].repeat(NUM_EPOCHS) # .repeat(): Function that repeats the data for the number of epochs (essential for streaming training), TF syntax
  val_dataset = None

else:
  val_dataset = rl_dataset[:int(NUM_VADIATION_DATA)]
  val_dataset = val_dataset.batch(EVAL_BATCH_SIZE).repeat(NUM_EPOCHS)

  train_dataset = rl_dataset[int(NUM_VADIATION_DATA): ]
  train_dataset = train_dataset.batch(TRAIN_MICRO_BATCH_SIZE).repeat(NUM_EPOCHS)



dataset_lengths = (
    len(train_dataset),
    len(val_dataset) if val_dataset is not None else 0,
)
print(f"dataset contains {dataset_lengths} of batches")
print(f"train dataset len: {len(train_dataset)}")

if val_dataset is not None:
    print(f"validation dataset len: {len(val_dataset)}")

## 2-4. Get SFT LoRA Adapter

In [ ]:
# We confirmed that in rare cases, errors related to NNX variables occur. This code is designed to handle that issue.

from flax.nnx import Variable
import functools



# Back up the original __setattr__ method.
_orig_setattr = Variable.__setattr__

# Define a new setattr function to address compatibility issues.
def _patched_setattr(self, name, value):
    # If attempting to set 'sharding_names',
    # handle it via set_metadata, which is the modern Flax approach.
    if name == 'sharding_names':
        self.set_metadata('sharding_names', value)
    else:
        # For other attributes, use the original method.
        _orig_setattr(self, name, value)

# Override the Variable class with the patched function.
Variable.__setattr__ = _patched_setattr

print("✅ Flax NNX Variable patched successfully for Qwix compatibility.")

In [ ]:
lora_policy = get_lora_model(ref_model, mesh=train_mesh)

sft_load_path = os.path.join(SFT_CKPT_DIR, "lora_params")
if not os.path.exists(sft_load_path):
    print(f"❌ WARNING: SFT checkpoint not found at: {sft_load_path}")
    print("   Using base model (fresh LoRA) without loading SFT params.")
else:
    checkpointer = ocp.StandardCheckpointer()

    with train_mesh:
        lora_state = nnx.state(lora_policy, nnx.LoRAParam)

        # CRITICAL: include train_mesh sharding so restore reshards to 4 devices
        shardings = nnx.get_named_sharding(lora_state, train_mesh)
        target = jax.tree_util.tree_map(
            lambda x, s: jax.ShapeDtypeStruct(x.shape, x.dtype, sharding=s),
            lora_state,
            shardings,
        )

        loaded_sft_params = checkpointer.restore(sft_load_path, target=target)

        nnx.update(
            lora_policy,
            jax.tree_util.tree_map(lambda _old, new: new, lora_state, loaded_sft_params),
        )

        # sync
        jax.tree_util.tree_map(
            lambda x: x.block_until_ready() if hasattr(x, "block_until_ready") else x,
            loaded_sft_params,
        )

    print("✓ SFT LoRA restored (resharded to train_mesh)")

nnx.display(lora_policy)

In [ ]:
del loaded_sft_params

gc.collect()
jax.clear_caches()

show_hbm_usage()

## 2-5. Reward Function

### 1) Format-check reward
We expect each model response to have the following structure:

`<reasoning_start> ... <reasoning_end>   (any text, non-greedy)
<solution_start>  ... <solution_end>    (we will capture the solution)
`

This regex allows leading/trailing whitespace and arbitrary text between blocks.
MULTILINE + DOTALL lets '.' match newlines and '^/$' work per line when needed.

In [ ]:
match_format = re.compile(
    rf"^[\s]*"
    rf"{re.escape(reasoning_start)}"
    rf".*?"
    rf"{re.escape(reasoning_end)}"
    rf".*?"
    rf"{re.escape(solution_start)}"
    rf"(.*?)"
    rf"{re.escape(solution_end)}"
    rf".*$",
    flags=re.MULTILINE | re.DOTALL,
)

def exact_format(prompts, completions, **kwargs):
    return [
        1.0 if match_format.search(resp) is not None else 0.0
        for resp in completions
    ]

### 2) Exact-answer reward
We extract the model’s final answer from the `<answer>...</answer>` block and assign a reward of `1.0` if the predicted answer matches the ground-truth numerically; otherwise `0.0`.

This numeric comparison avoids common false positives from substring matching (e.g., `“1”` vs. `“10”`) and is well-suited for math or short-answer tasks under tight runtime constraints.

For non-verifiable tasks without a unique ground-truth answer, this component typically yields a reward of `0.0`; however, this does not pose an issue, as the `total_reward` function appropriately normalizes and balances such cases across reward components.

In [ ]:
def extract_answer(text: str) -> str:
    # extract answer between <answer>...</answer>
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.DOTALL)
    if m:
        return m.group(1).strip()
    return text.strip()  # fallback: use full text

def exact_answer(prompts, completions, **kwargs):
    questions = kwargs.get("question", prompts)
    answers = kwargs.get("answer", kwargs.get("output"))
    if answers is None:
        raise ValueError("answer_reward requires `answer` in kwargs (dataset must provide it).")

    answers = list(answers)
    assert len(answers) == len(completions) == len(questions)

    rewards = []
    for ans, comp in zip(answers, completions):
        if ans is None or ans == "":
            rewards.append(0.0)
            continue

        pred = extract_answer(comp)

        try:
            rewards.append(1.0 if float(ans) == float(pred) else 0.0)
        except ValueError:
            rewards.append(0.0)

    return rewards

### 3) G-RaR (G-Eval + Rubrics-as-Rewards)
We propose **G-RaR**, a hybrid reward mechanism that integrates G-Eval with rubric-based rewards.

For each example, we first identify its task type (e.g., `Math`, `Coding`, `Summarization`) and retrieve the corresponding rubric and evaluation metric. To enable GRPO training under strict runtime constraints, task types and rubrics are pre-generated, and for each task, we select a single evaluation metric that captures the most critical aspect of performance.

Similar to G-Eval, we transform discrete rubric ratings into **a continuous-valued reward** to ensure stable learning. Specifically, we utilize the judge model’s logits over the discrete rating tokens `{1, 2, 3, 4, 5}`, compute a softmax distribution, and derive a continuous score via expectation, which is subsequently normalized to `[0, 1]`.

In [ ]:
# Task-specific judging prompt templates (AutoRubric)
# G-Eval refers to this as AutoCoT; since we use Rubrics-as-Rewards (RaR),
# we name it AutoRubric instead.

AUTORUBRIC_PROMPT = {
    "Creative writing": """You will be given a piece of creative writing generated in response to a prompt.
Your task is to evaluate the quality of the writing based on one specific metric.
Please read the text carefully and assess it according to the evaluation criteria below.
Focus only on the quality of the writing itself, not on personal preference or the topic choice.

Evaluation Criteria:
CraftQuality (1–5) – the overall craftsmanship of the creative writing.
This includes clarity of expression, narrative coherence, stylistic consistency, effective use of language, and how well the text conveys ideas or emotions in a polished and engaging manner.
A high score indicates writing that is well-structured, fluent, and intentionally crafted, rather than rough, confusing, or poorly organized.

Rubrics:""",

                  "Creative ideation": """You will be given a set of ideas generated in response to a creative ideation prompt.
Your task is to evaluate the diversity of the ideas.
Please carefully review all ideas as a group and assess how varied they are in perspective, approach, and content.

Evaluation Criteria:
Diversity (1–5) – the degree to which the generated ideas are distinct from one another.
This includes variation in themes, viewpoints, strategies, or conceptual directions.
A higher score reflects ideas that explore meaningfully different angles rather than repeating similar concepts with minor wording changes.

Rubrics:""",

                  "Summarization": """You will be given one summary written for a news article.
Your task is to rate the summary on one metric.
Please make sure you read and understand these instructions carefully. Please keep this
document open while reviewing, and refer to it as needed.

Evaluation Criteria:
Coherence (1-5) - the collective quality of all sentences. We align this dimension with
the DUC quality question of structure and coherence whereby ”the summary should be
well-structured and well-organized. The summary should not just be a heap of related information, but should build from sentence to sentence to a coherent body of information about a topic.”

Rubrics:""",

                  "Math": """You will be given a math problem along with a solution produced by a model.
Your task is to evaluate the correctness of the reasoning used to arrive at the solution.
Please carefully examine each step of the reasoning process.

Evaluation Criteria:
ReasoningCorrectness (1–5) – the logical validity and mathematical correctness of the reasoning steps.
A score of 5 indicates that all reasoning steps are valid, logically connected, and mathematically sound.
Lower scores indicate the presence of logical gaps, incorrect assumptions, or mathematical errors, even if the final answer happens to be correct.

Rubrics:""",

                  "Coding": """You will be given a programming task description and a piece of code generated to solve it.
Your task is to evaluate whether the code correctly fulfills the functional requirements of the task.

Evaluation Criteria:
FunctionalCorrectness (1–5) – the extent to which the code correctly implements the required functionality.
This includes correct handling of inputs, producing correct outputs, and satisfying edge cases described or implied by the task.
A higher score indicates code that would work as intended when executed, while a lower score reflects logical errors, missing functionality, or incorrect behavior.

Rubrics:""",

                  "Basic science": """You will be given a response to a basic science question.
Your task is to evaluate the scientific correctness of the response.
Please base your evaluation on established scientific knowledge and principles.

Evaluation Criteria:
ScientificCorrectness (1–5) – the accuracy and validity of the scientific statements made in the response.
A high score indicates that the explanation is factually accurate, scientifically sound, and free of misconceptions.
Lower scores indicate the presence of incorrect facts, flawed explanations, or misunderstandings of basic scientific concepts.

Rubrics:""",

                  "Other": """You will be given a response generated for a general reasoning task.
Your task is to evaluate the quality of the reasoning used in the response.
Carefully assess how well the response draws conclusions from the given information.

Evaluation Criteria:
GeneralReasoning (1–5) – the clarity, logical consistency, and soundness of the reasoning process.
A score of 5 indicates reasoning that is clear, well-structured, and logically valid.
Lower scores reflect unclear logic, unsupported conclusions, or reasoning errors.

Rubrics:"""
                 }

# Task → metric mapping
METRICS = {"Creative writing": "CraftQuality",
           "Creative ideation": "Diversity",
           "Summarization": "Coherence",
           "Math": "ReasoningCorrectness",
           "Coding": "FunctionalCorrectness",
           "Basic science": "ScientificCorrectness",
           "Other": "GeneralReasoning"
           }

In [ ]:
# utils

# score utilities
def get_token_ids_for_digits(judge_tokenizer):
    ids = {d: set() for d in ["1","2","3","4","5"]}   # collect token ids for digits 1–5
    variants = ["{}", " {}", "\n{}"]                 # common formatting variants
    for d in ids:
        for v in variants:
            tok = judge_tokenizer.encode(v.format(d))
            for t in tok:
                ids[d].add(int(t))                   # multiple tokens may map to one digit
    return ids


def build_grar_judge_prompt(
    question: str,
    completion: str,
    task_type: str,
    rubrics: str,
) -> str:
    autorubric_prompt = AUTORUBRIC_PROMPT[task_type]  # task-specific rubric instructions
    task_metrics = METRICS[task_type]                 # metric name for the task

    return (
        f"{autorubric_prompt}\n"
        f"{rubrics}\n\n"
        f"Question: {question}\n\n"
        f"Answer: {completion}\n\n"
        f"Output ONLY a single digit from 1 to 5.\n"  # enforce digit-only output
        f"Do NOT output words, explanations, punctuation, or extra tokens.\n"
        f"If you output anything other than one digit, the answer is invalid.\n"
        f"{task_metrics}: "
    )


def digit_only_softmax_from_logits(logits_last, digit_ids):
    # logits_last: (vocab,)
    ds = []
    for d in ["1","2","3","4","5"]:
        ids = jnp.array(sorted(list(digit_ids[d])))
        # merge multiple token variants for the same digit via log-sum-exp
        ds.append(jax.scipy.special.logsumexp(logits_last[ids]))
    ds = jnp.array(ds)          # (5,)
    pd = jax.nn.softmax(ds)     # (5,)
    return {str(i+1): float(pd[i]) for i in range(5)}  # digit -> probability


# bucketing to avoid recompilation
BUCKET_SIZES = (256, 512, 1024, 2048, 4096)  # adjust to judge model context
_mask_cache = {}  # (batch_size, bucket_len) -> cached causal mask

def pick_bucket_len(max_len: int) -> int:
    for b in BUCKET_SIZES:
        if max_len <= b:
            return b
    return BUCKET_SIZES[-1]

def get_causal_mask(batch_size: int, L: int) -> jnp.ndarray:
    key = (batch_size, L)
    m = _mask_cache.get(key)
    if m is None:
        m = jnp.tril(jnp.ones((batch_size, L, L), dtype=jnp.bool_))
        _mask_cache[key] = m
    return m

In [ ]:
digit_ids = get_token_ids_for_digits(judge_tokenizer)  # digit -> token id set
autorubric_cache = {}
task_type_cache = {}
debug_counter = 0

def g_rar(prompts, completions, **kwargs):
    global debug_counter
    debug_full = debug_counter < 3
    debug_top10 = True
    debug_counter += 1

    questions = list(kwargs["question"])
    # [Step 1] pregenerated task types
    task_types = list(kwargs["task_type"])
    # [Step 2] pregenerated rubrics
    rubrics = list(kwargs["rubrics"])

    batch_size = len(questions)

    # sanity checks (fail fast if dataset is misaligned)
    assert len(completions) == batch_size
    assert len(task_types) == batch_size
    assert len(rubrics) == batch_size

    # [Step 3] LLM-as-a-judge
    # build judge prompts (G-RaR)
    grar_prompts = []
    for q, comp, t_type, rub in zip(questions, completions, task_types, rubrics):
        grar_prompts.append(build_grar_judge_prompt(q, comp, t_type, rub))

    tokenized_list = [judge_tokenizer.encode(p) for p in grar_prompts]

    # bucketing + left padding
    max_len = max(len(t) for t in tokenized_list)
    bucket_len = pick_bucket_len(max_len)

    try:
        pad_id = judge_tokenizer.pad_id()
    except:
        pad_id = 0

    input_ids_np = np.full((batch_size, bucket_len), pad_id, dtype=np.int32)

    for i, tokens in enumerate(tokenized_list):
        if len(tokens) > bucket_len:
            tokens = tokens[-bucket_len:]
        input_ids_np[i, -len(tokens):] = tokens

    input_ids = jnp.array(input_ids_np)

    attention_mask = get_causal_mask(batch_size, bucket_len)
    positions = jnp.arange(bucket_len)[None, :]

    with judge_mesh:
        logits, _ = judge_model(
            last_tokens=input_ids,
            positions=positions,
            cache=None,
            attention_mask=attention_mask,
        )

    logits.block_until_ready()

    last_token_logits = logits[:, -1, :]  # logits at the final position
    rewards = []

    # compute expected score from digit probabilities
    for i in range(batch_size):
        p = digit_only_softmax_from_logits(last_token_logits[i], digit_ids)
        grar_score = sum(p[str(j)] * j for j in range(1, 6))  # expected score in [1, 5]
        reward = (grar_score - 1.0) / 4.0                     # normalize to [0, 1]
        rewards.append(reward)


    # === Uncomment the code below to print the logs ===

    # probs_top10 = jax.nn.softmax(last_token_logits[0], axis=-1) if debug_top10 else None

    # # debug: top-10 token probabilities for the first sample
    # if debug_top10 and probs_top10 is not None:
    #     top_ids = jnp.argsort(probs_top10)[-10:][::-1]
    #     top_tokens = [
    #         (judge_tokenizer.decode([int(t)]), float(probs_top10[int(t)]))
    #         for t in top_ids
    #     ]
    #     print("[g_rar] top10 tokens:", top_tokens)

    # # verbose debug output (first few calls only)
    # if debug_full:
    #     i = 0
    #     print("===== g_rar DEBUG =====")
    #     print(f"Q (prefix): {questions[i][:80]}...")
    #     print(f"C (prefix): {completions[i][:80]}...")
    #     print(f"Task Type: {task_types[i]}")
    #     print(f"AutoRubric (prefix): {rubrics[i][:100]}...")
    #     print(f"bucket_len: {bucket_len} (max_len was {max_len})")

    #     p_first = digit_only_softmax_from_logits(last_token_logits[0], digit_ids)
    #     print("p(1..5):", {k: round(v, 4) for k, v in p_first.items()})

    #     print(f"grar_score: {sum(p_first[str(j)] * j for j in range(1, 6)):.4f}")
    #     print(f"reward: {rewards[0]:.4f}")
    #     print("========================")

    #     print(
    #         f"[g_rar] batch mean={sum(rewards)/len(rewards):.3f}, "
    #         f"min={min(rewards):.3f}, max={max(rewards):.3f}"
    #     )

    return rewards

## 2-6. Total reward
We combine multiple reward components—`format correctness`, `exact answer correctness`, and `G-RaR`—using a weighted sum.

Each component captures a different aspect of quality:
- **Exact-format** enforces structural validity.
- **Exact-answer** provides a hard correctness signal when a unique ground-truth exists.
- **G-RaR** supplies a soft, task-aware quality score via rubric-based judging.

The final reward is optionally normalized by the sum of weights and clipped to `[0, 1]` for stability.

In [ ]:
def make_total_reward(
    *,
    weights=None,
    normalize_by_weight_sum=True):
    if weights is None:
        weights = {"exact_format": 0.2, "exact_answer": 0.2, "g_rar": 0.6}  # default heuristic weights

    w_format = float(weights.get("exact_format", 0.0)) # format reward weight
    w_answer = float(weights.get("exact_answer", 0.0)) # exact-answer reward weight
    w_grar  = float(weights.get("g_rar", 0.0)) # G-RaR reward weight

    w_sum = w_format + w_answer + w_grar
    if normalize_by_weight_sum and w_sum <= 0:
        raise ValueError("Sum of weights must be > 0 when normalize_by_weight_sum=True")

    def total_reward(prompts, completions, **kwargs):
        # format correctness
        r_format = np.asarray(exact_format(prompts, completions, **kwargs), dtype=np.float32)
        # exact-answer correctness
        r_answer = np.asarray(exact_answer(prompts, completions, **kwargs), dtype=np.float32)
        # G-RaR
        r_grar  = np.asarray(g_rar(prompts, completions, **kwargs), dtype=np.float32)

        # weighted sum of reward components
        total = (w_format * r_format) + (w_answer * r_answer) + (w_grar * r_grar)

        if normalize_by_weight_sum:
            total = total / w_sum # normalize to keep scale consistent
        total = np.clip(total, 0.0, 1.0) # safety clipping
        return total.tolist()

    return total_reward


In [ ]:
total_reward = make_total_reward(weights={"exact_format": 0.2, "exact_answer": 0.2, "g_rar": 0.6},
                   normalize_by_weight_sum=True)

## 2-7. Evaluate

In [ ]:
# Sampler for valiation
valid_sampler = sampler_lib.Sampler(
    transformer=lora_policy,
    tokenizer=ref_tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=TRAINER_MAX_PROMPT_LENGTH + TRAINER_TOTAL_GENERATION_STEPS + 256,
        num_layers=ref_model_config.num_layers,
        num_kv_heads=ref_model_config.num_kv_heads,
        head_dim=ref_model_config.head_dim,
    ),
)

In [ ]:
def valid_generate(
    prompts, sampler=valid_sampler, temperature=0.7, top_k=50, top_p=0.95, seed=None, max_generation_steps=TRAINER_TOTAL_GENERATION_STEPS
):
    """Given prompt, generates text."""

    if isinstance(prompts, str): # If it's a single string, wrap it in a list to create a batch.
        input_batch = [
            prompts
        ]
    else: # If it's already a list, format each prompt to create a new list.
        input_batch = prompts

    # Call the sampler function (model executor) to retrieve results.
    with train_mesh:
        out_data = sampler(
            input_strings=input_batch,       # List of formatted prompts
            max_generation_steps=max_generation_steps, # Maximum number of generated tokens
            temperature=temperature,         # Controls creativity/randomness
            top_k=top_k,
            top_p=top_p,
            echo=False,                      # Whether to include the input prompt in the output
            seed=seed if seed is not None else None, # Seed value for reproducibility
        )

    output = out_data.text

    if isinstance(prompts, str):
        return output[0]
    return output

evaluate function performs model inference on a given dataset to comprehensively evaluate both Exact Match accuracy and Judge model-based reward scores (LLM-as-a-Judge). It optimizes efficiency by batching generation and reward calculations, while ensuring precise answer verification and format compliance via numeric/string conversion logic. Additionally, it provides functionality to selectively extract and return correct or incorrect cases for in-depth error analysis based on the configuration.evaluate

In [ ]:
def evaluate(
    dataset,
    valid_sampler,
    reward_fn=None,  
    reward_threshold=0.8,  
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    corr_lst=False,  
    make_lst=False,  
):
    """Computes accuracy and percentage of outputs matching the format (Single Pass Optimized)."""

    response_lst = []
    corr = 0  
    llm_as_judge_corr = 0  
    llm_as_judge_sum = 0
    corr_format = 0  
    total = 0  

    # Helper function for numeric validation
    def is_numeric(s):
        """Checks if the string can be converted to a number."""
        try:
            float(s.strip())
            return True
        except (ValueError, AttributeError):
            return False

    for batch in tqdm(dataset):
        answers = batch["output"]  # List of ground truth answers
        questions = batch["question"]  # List of questions
        prompts = batch["prompts"]

        # Retrieve category info from the dataset
        categories = batch.get("category", ["Math"] * len(questions))
        task_types = batch["task_type"]
        rubrics = batch["rubrics"]

        responses = valid_generate(
            prompts, valid_sampler, temperature, top_k, top_p, seed=0
        )

        # Calculate the reward function first for all generated responses
        batch_rewards = [0.0] * len(responses)

        if reward_fn is not None:
            batch_rewards = reward_fn(
                prompts=prompts,
                completions=responses,
                question=questions,
                output=answers,
                task_type=task_types,
                rubrics=rubrics
            )
            # print(batch_rewards)

        # Scoring and Statistics Aggregation
        for i, (prompt, question, response, answer, category, task_type, rubric, reward) in enumerate(zip(
            prompts, questions, responses, answers, categories, task_types, rubrics, batch_rewards
        )):
            is_correct = False # Flag for current sample correctness

            # Check Format
            if match_format.search(response) is not None:
                corr_format += 1

            # Check Partial Match (Based on Reward Score)
            if reward >= reward_threshold:
                llm_as_judge_corr += 1

            # Accumulate LLM-as-judge score
            llm_as_judge_sum += reward

            # Extract Answer and Check Exact Match
            extracted_response = extract_answer(response)

            if extracted_response and answer:
                answer_is_numeric = is_numeric(str(answer))
                response_is_numeric = is_numeric(extracted_response)

                if answer_is_numeric and response_is_numeric:
                    if float(extracted_response.strip()) == float(str(answer).strip()):
                        is_correct = True
                elif str(extracted_response).strip() == str(answer).strip():
                    is_correct = True

            if is_correct:
                corr += 1

            # Save List (For error analysis, etc.)
            if make_lst:
                # If corr_lst is True, save correct cases; otherwise, save incorrect cases
                if (corr_lst and is_correct) or (not corr_lst and not is_correct):
                    response_lst.append((question, answer, response)) # response is now a str, not a list

            total += 1
            if total % 10 == 0:
                print(
                    f"===> {corr=}, {total=}, accuracy={corr / total * 100:.2f}%, "
                    f"llm_as_judge_corr={llm_as_judge_corr / total * 100:.2f}%, "
                    f"llm_as_judge_score_mean = {llm_as_judge_sum / total :.2f}, "
                    f"format={corr_format / total * 100:.2f}%"
                )

    to_return = (
        corr,
        total,
        corr / total * 100 if total > 0 else 0,
        llm_as_judge_corr / total * 100 if total > 0 else 0,
        llm_as_judge_sum / total if total > 0 else 0,
        corr_format / total * 100 if total > 0 else 0,
    )

    if make_lst:
        return to_return, response_lst

    return to_return

In [ ]:
# The evaluation might take up to couple of minutes to finish. Please be patient.
(corr, total, accuracy, llm_as_judge_accuracy,llm_as_judge_mean, format_accuracy) = evaluate(
    val_dataset,
    valid_sampler,
    reward_fn = g_rar,
    **GENERATION_CONFIGS["greedy"],
    reward_threshold=0.8,
)
print(
    f"{corr=}, {total=}, {accuracy=}%, {llm_as_judge_accuracy=}%, "
    f"{llm_as_judge_mean=}, {format_accuracy=}%"
)

In [ ]:
show_hbm_usage()

## 2-8. Train

In [ ]:
# Ckpt saving
checkpointing_options = ocp.CheckpointManagerOptions(
    save_interval_steps=SAVE_INTERVAL_STEPS, max_to_keep=MAX_TO_KEEP
)

# Metrics logger
metrics_logging_options = metrics_logger.MetricsLoggerOptions(
    log_dir="/tmp/content/tmp/tensorboard/grpo", flush_every_n_steps=20
)

In [ ]:
optimizer = optax.adamw(
    learning_rate=optax.schedules.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        decay_steps=MAX_STEPS,
        end_value=0.0,
    ),
    b1=B1,
    b2=B2,
    weight_decay=WEIGHT_DECAY,
)


if MAX_GRAD_NORM is not None:
  optimizer = optax.chain(
      optax.clip_by_global_norm(max_norm=MAX_GRAD_NORM),
      optimizer,
  )

In [ ]:
cluster_config = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: train_mesh, # ACTOR: The policy model being trained
        rl_cluster_lib.Role.REFERENCE: train_mesh, # REFERENCE: Frozen reference model (not updated during training)
        rl_cluster_lib.Role.ROLLOUT: train_mesh, # ROLLOUT: Responsible for generating responses from inputs (Inference)
    },
    rollout_engine='vanilla', # 'vanilla': Uses basic autoregressive generation without special search algorithms (like Tree Search)
    offload_to_cpu=False, # Keep all data/ops on accelerators (TPU/GPU) without offloading to CPU to maximize speed

    training_config=rl_cluster_lib.RLTrainingConfig(
        actor_optimizer=optimizer,
        eval_every_n_steps=EVAL_EVERY_N_STEPS,
        max_steps=MAX_STEPS,
        mini_batch_size=TRAIN_MICRO_BATCH_SIZE,
        train_micro_batch_size=TRAIN_MICRO_BATCH_SIZE,

        # metrics logging
        metrics_logging_options=metrics_logging_options,

        # checkpoint saving
        checkpoint_root_directory=RL_CKPT_DIR,
        checkpointing_options=checkpointing_options,
    ),

    rollout_config=base_rollout.RolloutConfig(
        max_tokens_to_generate=TRAINER_TOTAL_GENERATION_STEPS,
        max_prompt_length=TRAINER_MAX_PROMPT_LENGTH,
        kv_cache_size= TRAINER_MAX_PROMPT_LENGTH + TRAINER_TOTAL_GENERATION_STEPS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
    ),
)

grpo_config = GRPOConfig(
    # [G]: Number of responses generated per prompt (Group size)
    num_generations=NUM_GENERATIONS,

    # [μ]: Number of training iterations per data batch
    num_iterations=NUM_ITERATIONS,
    beta=BETA, # KL penalty coefficient
    epsilon=EPSILON, # [ε]: PPO clipping coefficient
    # """
    # GSPO-token is known for better training stability.
    # However, we observed that the loss is displayed as 0 when using GSPO-token.

    # So we used GRPO loss, which is the default setting.
    # Below are the references related to GSPO:
    # https://arxiv.org/abs/2507.18071
    # """
    #loss_algo='gspo-token'
)

In [ ]:
rl_cluster = rl_cluster_lib.RLCluster(
    actor=lora_policy,
    reference=lora_policy,
    tokenizer=ref_tokenizer,
    cluster_config=cluster_config,
)
grpo_trainer = GRPOLearner(
    rl_cluster=rl_cluster,
    reward_fns=total_reward,
    grpo_config=grpo_config,
)


In [ ]:
with train_mesh:
    grpo_trainer.train(train_dataset)

In [ ]:
wandb.init(project="tunix")

os.makedirs(RL_CKPT_DIR, exist_ok=True)

rl_lora_state = nnx.state(lora_policy, nnx.LoRAParam)
print(f"\nNumber of LoRA parameters to save: {len(jax.tree.leaves(rl_lora_state))}")

rl_checkpointer = ocp.StandardCheckpointer()

# Save LoRA Adapter
rl_save_path = os.path.join(RL_CKPT_DIR, "lora_params")
print(f"\nSaving model to: {rl_save_path}")
rl_checkpointer.save(rl_save_path, rl_lora_state,force=True)

rl_checkpointer.wait_until_finished()

print("✓ RL model saved successfully!")
print(f"   Save path: {rl_save_path}")

# Load the final checkpoint for single-session mode evaluation

In [ ]:

trained_ckpt_path = os.path.join(RL_CKPT_DIR, "lora_params")

print(f"Loading checkpoint from: {trained_ckpt_path}")

abs_params = jax.tree.map(
    lambda x: jax.ShapeDtypeStruct(x.shape, x.dtype),
    nnx.state(lora_policy, nnx.LoRAParam),
)
checkpointer = ocp.StandardCheckpointer()
trained_lora_params = checkpointer.restore(trained_ckpt_path, target=abs_params)

nnx.update(
    lora_policy,
    jax.tree.map(
        lambda a, b: b,
        nnx.state(lora_policy, nnx.LoRAParam),
        trained_lora_params,
    ),
)

In [ ]:
# The evaluation might take up to couple of minutes to finish. Please be patient.
(corr, total, accuracy, llm_as_judge_accuracy,llm_as_judge_mean, format_accuracy) = evaluate(
    val_dataset,
    valid_sampler,
    reward_fn = g_rar,
    **GENERATION_CONFIGS["greedy"],
    reward_threshold=0.8,
)
print(
    f"{corr=}, {total=}, {accuracy=}%, {llm_as_judge_accuracy=}%, "
    f"{llm_as_judge_mean=}, {format_accuracy=}%"
)